# Config

In [1]:
VERBOSE = True
DEBUG = False
PICKLE_VER = 2 # 1 - original version, 2 - version with GraphDataLibraryNoEmbedding converted to dict
GRAPH_SOURCE = 'directed' # directed, undirected graph data for building cluster (louvain, leiden)
CLUSTER_METHOD = 'louvain'

In [2]:
if DEBUG:
    from models import *
    from gen_cluster_mitigationplan import *
    # import os, pickle
    # from .core import *

    dir_path = os.getcwd()
    # pcg_data_path = f"{dir_path}/graph_data_library_no_embedding.pkl"
    pcg_data_path = f"{dir_path}/graph_data_library_no_embedding_dict.pkl"
    # pcg_data_path = f"{dir_path}/graph_data_library_no_embedding_dict_v2.pkl"
    # pcg_data_path = f"{dir_path}/graph_data_library.pkl"
    # graph_data_library.pkl
    # data = pickle.load(open(pcg_data_path, "rb"))

    # data: GraphDataLibraryNoEmbedding = pickle.load(open(pcg_data_path, "rb"))

    data_all = pickle.load(open(pcg_data_path, "rb"))
    # data_dict = data.model_dump()

    company_name = 'PCG|embedding_risk_desc_catalog|oneway_run'
    data = data_all['company_graph_datas'][company_name]
    debug_list = []
    all_data_list = []
# for company, data in company_graph_datas.items():
    G = nx.DiGraph()
    print(data.keys())
    nodes = data["nodes"]
    edges = data["edges"]
    print(edges[0].keys())
    line_weights = [edge["cosine_similarity"] for edge in edges]
    num_edges_to_show = get_number_edges_to_show(len(nodes))
    sorted_weights = sorted(line_weights, reverse=True)
    # The threshold is the weight of the (num_edges_to_show)-th edge (0-indexed)
    slider_value = sorted_weights[num_edges_to_show - 1]
    filter_edges = filter_non_arrow_edges2(edges, slider_value)
    for edge in filter_edges:
        G.add_edge(
            edge["source"],
            edge["target"],
            weight=edge["cosine_similarity"],
        )
        if edge["source"] == "risk_PCG_40":
            debug_list.append(edge)
    in_degree_centrality_dict = nx.in_degree_centrality(G)
    out_degree_centrality_dict = nx.out_degree_centrality(G)
    betweenness_dict_weight = nx.betweenness_centrality(G, weight="weight")
    betweenness_dict_non_weight = nx.betweenness_centrality(G)

    # create list of data so I can convert to dataframe later
    for node in nodes:
        row_data = {
            # "company": company,
            "risk_id": node["data"]["id"],
            "risk_name": node["data"]["label"],
            "risk_level": node["data"]["risk_level"],
            "in_degree": G.in_degree(node["data"]["id"]),
            "out_degree": G.out_degree(node["data"]["id"]),
            "in_degree_centrality": in_degree_centrality_dict.get(
                node["data"]["id"], None
            ),
            "out_degree_centrality": out_degree_centrality_dict.get(
                node["data"]["id"], None
            ),
            "betweenness_centrality_weight": betweenness_dict_weight.get(
                node["data"]["id"], None
            ),
            "betweenness_centrality_non_weight": betweenness_dict_non_weight.get(
                node["data"]["id"], None
            ),
        }
        all_data_list.append(row_data)

    all_data_df = pd.DataFrame(all_data_list)
    all_data_df

In [3]:
# edges

In [4]:
# filter_edges

In [5]:
# data_all['company_graph_datas']['PCG|embedding_risk_desc_catalog|oneway_run']['edges']

In [6]:
# data_all['company_graph_datas']['PCG|embedding_risk_desc_catalog|oneway_run']['nodes']

# load current version of pickle graph data

In [7]:
if PICKLE_VER == 2:
    from models import *
    from gen_cluster_mitigationplan import *
    # import os, pickle
    # from .core import *

    dir_path = os.getcwd()
    # pcg_data_path = f"{dir_path}/graph_data_library_no_embedding.pkl"
    # pcg_data_path = f"{dir_path}/graph_data_library_no_embedding_dict.pkl"
    pcg_data_path = f"{dir_path}/graph_data_library_no_embedding_dict_v2.pkl"
    data = pickle.load(open(pcg_data_path, "rb"))
    data_source_selected = 'PCG|embedding_risk_desc_catalog|oneway_run' # 'lotus_south|embedding_risk_desc_catalog|oneway_run'
    all_data_df, clusters_directed, G, nodes = find_graph_properties_newpickle(pcg_data_path, data_source_selected)
    
    # all_data_df, clusters, G, nodes = find_graph_properties_newpickle(pcg_data_path, data_source_selected)

    pcg_data_path2 = f"{dir_path}/graph_data_library_no_embedding_dict_v2.pkl" # result in smaller cluster of 'high' risk for PCG   
    data2 = pickle.load(open(pcg_data_path2, "rb"))
    all_data_df2, clusters_directed2, G2, nodes2 = find_graph_properties_newpickle(pcg_data_path2, data_source_selected)

    # clusters_undirected = find_clusters_from_graph(pcg_data_path, data_source_selected,CLUSTER_METHOD = 'leiden',graph_dir='undirected')
    # clusters_directed = find_clusters_from_graph(pcg_data_path, data_source_selected,CLUSTER_METHOD = 'leiden',graph_dir='directed')
    clusters_undirected = find_clusters_from_graph(pcg_data_path, data_source_selected,CLUSTER_METHOD = CLUSTER_METHOD,graph_dir='undirected')
    clusters_directed = find_clusters_from_graph(pcg_data_path, data_source_selected,CLUSTER_METHOD = CLUSTER_METHOD,graph_dir='directed')

    if GRAPH_SOURCE == 'undirected':
        clusters = clusters_undirected
    else:
        clusters = clusters_directed


In [8]:
# def find_clusters_from_graph(data_path, company_name, CLUSTER_METHOD = 'leiden', graph_dir = 'directed'): # sink/source/central nodes; clusters

# CLUSTER_METHOD = 'leiden'
# graph_dir='directed'

data_all = pickle.load(open(pcg_data_path, "rb"))

# data = data_all['company_graph_datas'][company_name]

all_data_list = []
# for i, company_name in enumerate(data_all['company_graph_datas'].keys()):
G = nx.DiGraph()

data = data_all['company_graph_datas'][data_source_selected]
# data = pickle.load(open(data_path, "rb"))

nodes = data['nodes']
edges = data['edges']
line_weights = [edge["cosine_similarity"] for edge in edges]
num_edges_to_show = get_number_edges_to_show(len(nodes))
sorted_weights = sorted(line_weights, reverse=True)
# The threshold is the weight of the (num_edges_to_show)-th edge (0-indexed)
slider_value = sorted_weights[num_edges_to_show - 1]
if GRAPH_SOURCE == 'directed':
    filter_edges = filter_non_arrow_edges2(edges, slider_value)
else:
    filter_edges = filter_none_edges(edges, slider_value)
for edge in filter_edges:
    G.add_edge(
        edge["source"],
        edge["target"],
        weight=edge["cosine_similarity"],
        # weight=(edge["cosine_similarity"]+1.0)/2.0, # shift weight since cosine similarity range is -1 and 1, but RBER only accepts positive values
    )
# === STEP 3: CLUSTER RISKS ===
# louvain
if CLUSTER_METHOD == 'louvain':
    clusters = louvain_communities(G)

# leiden
if CLUSTER_METHOD == 'leiden':
    # clusters = leiden_communities(G, backend="cugraph") # this method needs cugraph backend
    # Step 3.2: Convert NetworkX to igraph
    # G_ig = ig.Graph.TupleList(G.edges(), directed=False)
    if GRAPH_SOURCE == 'directed':
        G_ig = ig.Graph.TupleList(G.edges(), directed=True, vertex_name_attr="name")
        G_ig.add_vertices(list(G.nodes()))
        G_ig.add_edges(list(G.edges()))
        weights = [G[u][v].get("weight", 1.0) for u, v in G.edges()]
        # weights = (weights + 1.0) / 2.0
        G_ig.es["weight"] = weights
    else:
        G_ig = ig.Graph.TupleList(G.edges(), directed=False, vertex_name_attr="name")

    # Step 3.3: Run Leiden algorithm
    # partition = leidenalg.find_partition(G_ig, leidenalg.CPMVertexPartition) # doesn't seem to work with directed graph
    # partition = leidenalg.find_partition(G_ig, leidenalg.RBERVertexPartition)
    partition = leidenalg.find_partition(G_ig, leidenalg.RBERVertexPartition, resolution_parameter=1.2, weights="weight",n_iterations=50)
    # partition = leidenalg.find_partition(G_ig, leidenalg.ModularityVertexPartition) # for non-directed

    print("Communities:", partition.membership)
    print("Quality:", partition.quality())

    # Step 3.4: Extract communities (as lists of original node names)
    clusters = [ [G_ig.vs[node]["name"] for node in community] for community in partition ]
    

In [9]:
import random
def run_leiden_RBER(g, gamma, seed, n_iter=20, weights=None):
    random.seed(seed)
    part = leidenalg.find_partition(
        g,
        leidenalg.RBERVertexPartition,
        resolution_parameter=gamma,
        weights=weights,
        n_iterations=n_iter,
    )
    return {
        "gamma": gamma,
        "seed": seed,
        "k": len(part),
        "quality": float(part.quality()),
        "membership": part.membership,
    }

def sweep_resolutions(
    g,
    gammas=(0.05, 0.1, 0.2, 0.4, 0.8, 1.2, 1.6, 2.0),
    seeds=(0,1,2,3,4),
    n_iter=30,
    weights=None,
):
    results = []
    for gamma in gammas:
        for s in seeds:
            results.append(run_leiden_RBER(g, gamma, s, n_iter, weights))
    return results

# results = sweep_resolutions(G_ig)

In [10]:
# pip install python-igraph leidenalg networkx scikit-learn

import math
import random
import statistics
from collections import defaultdict

import networkx as nx
import igraph as ig
import leidenalg as la
from sklearn.metrics import normalized_mutual_info_score as NMI


# ----------------------------
# 1) Weight handling utilities
# ----------------------------

def transform_weight(w: float, mode: str = "truncate") -> float:
    """
    Convert weights from [-1, 1] to non-negative for Leiden.
    - 'truncate': negatives -> 0 (treat as no tie)
    - 'shift01' : w' = (w + 1) / 2  in [0,1]
    - 'abs'     : magnitude only
    """
    if mode == "truncate":
        return w if w > 0 else 0.0
    if mode == "shift01":
        return (w + 1.0) / 2.0
    if mode == "abs":
        return abs(w)
    raise ValueError("transform mode must be 'truncate', 'shift01', or 'abs'")


def normalize_weights(ws, scheme: str = "median1"):
    """
    Bring weights to a comparable scale so no single edge dominates.
    - None      : no normalization
    - 'max1'    : divide by max -> max=1
    - 'median1' : divide by median -> median≈1
    """
    if not ws:
        return ws
    if scheme is None:
        return ws
    if scheme == "max1":
        m = max(ws)
        return [ (w / m) if m > 0 else 0.0 for w in ws ]
    if scheme == "median1":
        m = statistics.median(ws)
        return [ (w / m) if m > 0 else 0.0 for w in ws ]
    raise ValueError("normalize scheme must be None, 'max1', or 'median1'")


# -----------------------------------------
# 2) Convert nx.DiGraph -> igraph with wts
# -----------------------------------------

def nx_to_igraph_with_weights(
    G: nx.DiGraph,
    weight_attr: str = "weight",
    transform: str = "shift01",
    normalize: str = "median1",
) -> ig.Graph:
    """
    Converts a NetworkX DiGraph (or Graph) to igraph, copying & transforming weights.
    Leaves zero-weight edges in place (Leiden effectively ignores them).
    """
    directed = G.is_directed()
    g = ig.Graph(directed=directed)

    nodes = list(G.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    g.add_vertices(len(nodes))
    g.vs["name"] = nodes

    edges = [(idx[u], idx[v]) for u, v in G.edges()]
    g.add_edges(edges)

    raw = [float(G[u][v].get(weight_attr, 1.0)) for u, v in G.edges()]
    trans = [transform_weight(w, transform) for w in raw]
    wts = normalize_weights(trans, normalize)
    g.es["weight"] = wts

    print(wts)

    return g


# --------------------------------
# 3) Run Leiden (RBER) once
# --------------------------------

def run_leiden_rber(
    g: ig.Graph,
    gamma: float,
    seed: int,
    n_iter: int = 30,
    weight_attr: str = "weight",
    initial_membership=None,
):
    """
    Single Leiden run with RBER objective.
    """
    random.seed(seed)
    part = la.find_partition(
        g,
        la.RBERVertexPartition,
        resolution_parameter=gamma,
        weights=weight_attr,
        n_iterations=n_iter,
        initial_membership=initial_membership,
    )
    return {
        "gamma": gamma,
        "seed": seed,
        "k": len(part),
        "quality": float(part.quality()),
        "membership": part.membership,  # list[int], aligned to g.vs indices
        "partition": part,
    }


# ------------------------------------------------
# 4) Grid sweep over resolutions and seeds
# ------------------------------------------------

def sweep_resolutions(
    g: ig.Graph,
    gammas=(0.05, 0.1, 0.2, 0.4, 0.8, 1.2, 1.6, 2.0),
    seeds=tuple(range(10)),  # 10 seeds default
    n_iter=30,
    weight_attr="weight",
):
    results = []
    for gamma in gammas:
        for s in seeds:
            results.append(run_leiden_rber(g, gamma, s, n_iter, weight_attr))
    return results


# ---------------------------------------------------
# 5) Stability (NMI) & gamma selection
# ---------------------------------------------------

def nmi_matrix(partitions):
    """
    Compute NMI for all pairs of partitions (same node set).
    Returns upper-triangle list and mean NMI.
    """
    n = len(partitions)
    if n <= 1:
        return [1.0], 1.0

    scores = []
    for i in range(n):
        for j in range(i + 1, n):
            a = partitions[i]["membership"]
            b = partitions[j]["membership"]
            scores.append(NMI(a, b))
    mean_nmi = sum(scores) / len(scores) if scores else 1.0
    return scores, mean_nmi


def summarize_by_gamma(results):
    """
    For each gamma, compute:
    - mean NMI across seeds
    - mean/stdev of community count k
    - mean quality
    Returns dict[gamma] -> summary + per-seed results
    """
    by_gamma = defaultdict(list)
    for r in results:
        by_gamma[r["gamma"]].append(r)

    summary = {}
    for gamma, rows in by_gamma.items():
        _, mean_nmi = nmi_matrix(rows)
        ks = [r["k"] for r in rows]
        q = [r["quality"] for r in rows]
        summary[gamma] = {
            "mean_nmi": mean_nmi,
            "mean_k": statistics.mean(ks),
            "std_k": statistics.pstdev(ks) if len(ks) > 1 else 0.0,
            "mean_quality": statistics.mean(q),
            "runs": rows,
        }
    return summary


def choose_best_gamma(summary, prefer_stability_weight=1.0, penalize_k_variance=0.25):
    """
    Score each gamma:
      score = prefer_stability_weight * mean_nmi
              + scaled(mean_quality)
              - penalize_k_variance * std_k_norm
    Uses z-ish scaling for quality and std_k to keep scales comparable.
    Returns (best_gamma, ranked_list)
    """
    gammas = list(summary.keys())
    mean_qualities = [summary[g]["mean_quality"] for g in gammas]
    std_ks = [summary[g]["std_k"] for g in gammas]

    # simple scaling to [0,1]
    def scale(vals):
        vmin, vmax = min(vals), max(vals)
        if vmax == vmin:
            return [0.5 for _ in vals]
        return [(v - vmin) / (vmax - vmin) for v in vals]

    q_scaled = scale(mean_qualities)
    stdk_scaled = scale(std_ks)

    scored = []
    for i, g in enumerate(gammas):
        s = summary[g]
        score = (
            prefer_stability_weight * s["mean_nmi"]
            + 0.5 * q_scaled[i]                 # modest tie-break on quality
            - penalize_k_variance * stdk_scaled[i]
        )
        scored.append((g, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    best_gamma = scored[0][0]
    return best_gamma, scored


# ---------------------------------------------------
# 6) Pick medoid partition (most representative run)
# ---------------------------------------------------

def medoid_partition(rows_at_gamma):
    """
    Select the run whose membership has the highest average NMI to others.
    """
    if len(rows_at_gamma) == 1:
        return rows_at_gamma[0]

    best = None
    best_avg = -1
    for i, r in enumerate(rows_at_gamma):
        nmis = []
        for j, s in enumerate(rows_at_gamma):
            if i == j:
                continue
            nmis.append(NMI(r["membership"], s["membership"]))
        avg_nmi = sum(nmis) / len(nmis) if nmis else 1.0
        if avg_nmi > best_avg:
            best = r
            best_avg = avg_nmi
    return best


# ---------------------------------------------------
# 7) Optional refine with warm-start
# ---------------------------------------------------

def refine_with_warm_start(g, gamma, base_membership, n_iter=60, seed=123):
    """
    Run Leiden again at the chosen gamma, warm-starting from medoid labels.
    """
    return run_leiden_rber(
        g,
        gamma=gamma,
        seed=seed,
        n_iter=n_iter,
        initial_membership=base_membership,
    )


# ---------------------------------------------------
# 8) End-to-end wrapper
# ---------------------------------------------------

def find_best_partition_from_nx(
    G: nx.DiGraph,
    weight_attr="weight",
    transform="shift01",     # or 'shift01' — try both and compare
    normalize="median1",
    gammas=(0.05, 0.1, 0.2, 0.4, 0.8, 1.2, 1.6, 2.0),
    seeds=tuple(range(10)),
    n_iter=30,
    warm_refine=True,
):
    """
    Full pipeline:
      - NX -> igraph with proper weights
      - sweep gamma × seed
      - pick best gamma by stability (mean NMI), tie-break by quality, penalize k variance
      - choose medoid partition at best gamma
      - optional warm-start refinement
      - return final labels mapped back to original node IDs + diagnostics
    """
    g = nx_to_igraph_with_weights(G, weight_attr, transform, normalize)
    results = sweep_resolutions(g, gammas, seeds, n_iter, "weight")
    summary = summarize_by_gamma(results)
    best_gamma, ranking = choose_best_gamma(summary)

    chosen_runs = summary[best_gamma]["runs"]
    medoid = medoid_partition(chosen_runs)

    final = medoid
    if warm_refine:
        final = refine_with_warm_start(
            g, gamma=best_gamma, base_membership=medoid["membership"], n_iter=max(60, n_iter)
        )

    # map membership back to original node names
    idx_to_name = g.vs["name"]
    membership = final["membership"]
    partition_from_loop = final["partition"]
    labels_by_node = {idx_to_name[i]: membership[i] for i in range(len(membership))}

    diagnostics = {
        "best_gamma": best_gamma,
        "ranking": ranking,                         # [(gamma, score), ...] high->low
        "summary": {g_: {k: (v if k != "runs" else None) for k, v in s.items()}
                    for g_, s in summary.items()},  # summary stats per gamma
        "medoid_seed": medoid["seed"],
        "medoid_k": medoid["k"],
        "final_k": len(set(membership)),
        "final_quality": float(final["quality"]),
        "transform": transform,
        "normalize": normalize,
        "n_iter": n_iter,
        "warm_refine": warm_refine,
    }
    return labels_by_node, diagnostics, membership, partition_from_loop


# ------------------------
# 9) Example usage
# ------------------------
# if __name__ == "__main__":
#     # Build a tiny demo DiGraph with weights in [-1, 1]
#     G = nx.DiGraph()
#     edges = [
#         (0, 1, 0.7), (1, 2, 0.6), (2, 0, 0.5),
#         (3, 4, 0.8), (4, 5, 0.7), (5, 3, 0.6),
#         (2, 3, -0.2), (5, 0, 0.3), (1, 4, -0.4),
#     ]
#     for u, v, w in edges:
#         G.add_edge(u, v, weight=w)

# Run full pipeline (try 'shift01' too and compare)
TEST_CLUSTER = False

if TEST_CLUSTER:
    labels, info, membership, partition_from_loop = find_best_partition_from_nx(
        G,
        weight_attr="weight",
        transform="shift01",      # or 'shift01'
        normalize="median1",
        gammas=(0.8, 1.2), # (0.05, 0.1, 0.2, 0.4, 0.8, 1.2),
        seeds=tuple(range(8)),
        n_iter=50,
        warm_refine=True,
    )

    print("Best gamma:", info["best_gamma"])
    print("Final communities (label by node):", labels)
    print("Final k:", info["final_k"], "quality:", info["final_quality"])

    clusters = [ [G_ig.vs[node]["name"] for node in community] for community in partition ]

In [11]:
# clusters

In [12]:
# results

In [ ]:
# weights

NameError: name 'weights' is not defined

In [ ]:
all_data_df

In [ ]:
all_data_df2.in_degree[0]

In [ ]:
all_data_df.out_degree.sum()

In [ ]:
all_data_df2.out_degree.sum()

In [ ]:
all_data_df2 == all_data_df

In [ ]:
clusters

In [ ]:
clusters_directed2

In [ ]:
for i, j in enumerate(data['company_graph_datas'].keys()):
    print(i)
    print(j)

# load original version of pickle graph data

In [ ]:
if PICKLE_VER == 1:
    from gen_cluster_mitigationplan import *
    from models import *

    # ==================== TEST CODE =====================
    dir_path = os.getcwd()
    # pcg_data_path = f"{dir_path}/graph_data_library_no_embedding.pkl"
    pcg_data_path = f"{dir_path}/nodes_and_edges_PCG.pkl"
    lotus_south_data_path = f"{dir_path}/nodes_and_edges_lotus_south.pkl"

    # data_path_dict = {"PCG": pcg_data_path, "Lotus South": lotus_south_data_path}
    data_path_dict = {"PCG": pcg_data_path}

    all_data_df, clusters, G, nodes = find_graph_properties(data_path_dict)


# find 'important' clusters

In [15]:
risk_high = all_data_df[all_data_df['risk_level']>=3] # TODO: need to handle the edge case when there's no high/critical risk level
# TopN risks with a given property (central/source/sink)
N_TOP = 3
# risk_central = top_n_with_row_limit(all_data_df, 'betweenness_centrality_non_weight', n=N_TOP)
# risk_source = top_n_with_row_limit(all_data_df, 'out_degree', n=N_TOP)
# risk_central = top_n_strict_with_priority_on_highest(all_data_df, 'betweenness_centrality_non_weight', n=N_TOP)
# risk_source = top_n_strict_with_priority_on_highest(all_data_df, 'out_degree', n=N_TOP)

risk_central = top_n_with_threshold(all_data_df, 'betweenness_centrality_non_weight', n=N_TOP, threshold=0.01)
risk_source = top_n_with_threshold(all_data_df, 'out_degree', n=N_TOP, threshold=2)

# if VERBOSE:
#     print('high risk: ' + ', '.join(risk_high['risk_name'].tolist()))
#     print('central risk: ' + ', '.join(risk_central['risk_name'].tolist()))
#     print('source risk: ' + ', '.join(risk_source['risk_name'].tolist()))

# ======= cluster by type (high/central/source risk) =======

# cluster(s) that contains risk_high
highrisk_clusters = find_sublists_with_any(risk_high['risk_id'].tolist(), clusters)
# print(highrisk_clusters)

# cluster(s) that contains risk_central
centralrisk_clusters = find_sublists_with_any(risk_central['risk_id'].tolist(), clusters)
# print(centralrisk_clusters)

# cluster(s) that contains risk_source
sourcerisk_clusters = find_sublists_with_any(risk_source['risk_id'].tolist(), clusters)
# print(sourcerisk_clusters)

if VERBOSE:
    print('high risk: ' + ', '.join(risk_high['risk_name'].tolist()))
    for i, cluster in enumerate(highrisk_clusters, 1):
        print(f"\n--- Cluster {i} Prompt ---\n")
        print(generate_prompt_nointro(cluster, G, nodes))

    print('central risk: ' + ', '.join(risk_central['risk_name'].tolist()))
    for i, cluster in enumerate(centralrisk_clusters, 1):
        print(f"\n--- Cluster {i} Prompt ---\n")
        print(generate_prompt_nointro(cluster, G, nodes))

    print('source risk: ' + ', '.join(risk_source['risk_name'].tolist()))
    for i, cluster in enumerate(sourcerisk_clusters, 1):
        print(f"\n--- Cluster {i} Prompt ---\n")
        print(generate_prompt_nointro(cluster, G, nodes))

# ======= ALL important clusters (contains high/central/source risks) =======
combined_list = list(set(risk_high['risk_id'].tolist() + risk_central['risk_id'].tolist() + risk_source['risk_id'].tolist()))
important_clusters = find_sublists_with_any(combined_list, clusters)

# # === STEP 5: OUTPUT PROMPTS ===
print('============= Important Clusters =============')
for i, cluster in enumerate(important_clusters, 1):
    print(f"\n--- Cluster {i} Prompt ---\n")
    print(generate_prompt_nointro(cluster, G, nodes))

# print(generate_prompt_nointro(important_clusters[0], G, nodes))

high risk: Intense market competition

--- Cluster 1 Prompt ---

Risks:
- risk_20250513_65: Wrong delivery
- risk_20250513_32: New competitor into the market
- risk_20250513_41: Poor service quality
- risk_20250513_24: Intense market competition
- risk_20250513_54: Service-related dissatisfaction
- risk_20250513_59: Uncompetitive service

Dependencies:
- risk_20250513_65 -> risk_20250513_54 
- risk_20250513_32 -> risk_20250513_24 
- risk_20250513_41 -> risk_20250513_54 
- risk_20250513_41 -> risk_20250513_59 
- risk_20250513_41 -> risk_20250513_65 
- risk_20250513_24 -> risk_20250513_59 
- risk_20250513_54 -> risk_20250513_59 

central risk: Unable to deliver product, Inventory damage, Operational inefficiency

--- Cluster 1 Prompt ---

Risks:
- risk_20250513_18: Hardware Failure
- risk_20250513_29: Machinery / equipment failure and damage
- risk_20250513_12: Facility infrastructure failure and damage
- risk_20250513_55: Software Failure
- risk_20250513_25: Inventory damage
- risk_2025

In [ ]:
risk_high

In [ ]:
all_data_df

# gen 'cluster' mitigation plan

In [ ]:
# from dotenv import load_dotenv
# import os
# from typing import List, Tuple

# TODO: need to acquire the api_key from your directory
# load_dotenv("../../.env")

estimate_cost.total_cost_THB = 0 # initialization
usage_count_list = []

# notice, this is a single cluster
cluster_risk_to_plan = generate_prompt_nointro(important_clusters[1], G, nodes)
# cluster_risk_to_plan = generate_prompt_nointro(highrisk_clusters, G, nodes)

# gen 'cluster' mitigation plans
response = get_response_control_cluster(cluster_risk_to_plan) # accept single cluster
# response = get_response_test(cluster_risk_to_plan)

usage_count = response.usage
usage_count_list.append(usage_count)    
MODEL = "gpt-4.1" # "gpt-4o"
estimate_cost(res_usage = usage_count_list,type='multi',model=MODEL)

In [ ]:
# response.choices[0].message.content

In [ ]:
# save to json
tmp_res = response.choices[0].message.content
report_json = json.loads(tmp_res)

data = []
data.append(report_json)

# input_file = report_json_folder + str(selected_year) + '_Q' + str(selected_quarter) + '_' + selected_company_report + '_json_riskcontrol.json'
file_path = f"{dir_path}/clustercontrol.json"

with open(file_path, 'w') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

report_json